# Capítulo 12: Métodos Experimentales de Laboratorio y Análisis en Python
### Física Electrónica - Cátedra UTN FRA
*Procesamiento computacional integral de los Trabajos Prácticos de Laboratorio:*
- **TP 1:** Propagación de incertidumbres y cálculo de errores con SymPy.
- **TP 3:** Carga específica del electrón ($e/m$) en bobinas de Helmholtz.
- **TP 4:** Espectrometría y determinación de la constante de Rydberg ($R_y$).
- **TP 5:** Experimento de Franck-Hertz y resonancia atómica a $4.9\text{ V}$.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)


## Práctica 1: Propagación de Incertidumbres en Cuadratura (TP 1)
Para una magnitud indirecta $f(x, y, z)$, la incertidumbre combinada se calcula simbólicamente mediante:
$$\Delta f = \sqrt{ \left(\frac{\partial f}{\partial x}\Delta x\right)^2 + \left(\frac{\partial f}{\partial y}\Delta y\right)^2 + \left(\frac{\partial f}{\partial z}\Delta z\right)^2 }$$

In [ ]:
import sympy as sp
import numpy as np

# Definición de variables simbólicas e incertezas
V, I = sp.symbols('V I')
dV, dI = sp.symbols('Delta_V Delta_I')

# Función de potencia: P = V * I
P = V * I
dP = sp.sqrt((sp.diff(P, V) * dV)**2 + (sp.diff(P, I) * dI)**2)

print('Expresión simbólica de la incertidumbre en Potencia:')
sp.pprint(dP)

# Evaluación numérica de ejemplo: V = 12.0 +- 0.1 V, I = 2.50 +- 0.05 A
valores = {V: 12.0, I: 2.50, dV: 0.1, dI: 0.05}
P_num = float(P.subs(valores))
dP_num = float(dP.subs(valores))
print(f'\nResultado: Potencia P = {P_num:.2f} +- {dP_num:.2f} W (Error relativo: {dP_num/P_num*100:.2f}%)')

## Práctica 2: Carga Específica del Electrón $e/m$ (TP 3)
A partir de la relación linealizada:
$$U = \frac{K^2}{2}\left(\frac{e}{m}\right) i^2 r^2$$
con $K = \mu_0 (4/5)^{3/2} N / R \approx 6.924 \times 10^{-4}\text{ T/A}$.

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

K_helm = 6.924e-4 # T / A

# Datos de laboratorio de ejemplo (U [V], i [A], r [m])
i_exp = np.array([1.2, 1.4, 1.6, 1.8, 2.0])
r_exp = np.array([0.04, 0.04, 0.04, 0.04, 0.04]) # radio de órbita fijado a 4 cm
U_exp = np.array([118.0, 160.0, 210.0, 265.0, 328.0])

x_data = (i_exp**2) * (r_exp**2)

def modelo_lineal(x, m_pendiente):
    return m_pendiente * x

popt, pcov = curve_fit(modelo_lineal, x_data, U_exp)
m_fit = popt[0]
e_m_calc = (2.0 * m_fit) / (K_helm**2)
e_m_teorico = 1.7588e11

print(f'Pendiente ajustada: {m_fit:.2e} V / (A^2 m^2)')
print(f'Carga específica e/m calculada: {e_m_calc:.3e} C/kg')
print(f'Valor teórico: {e_m_teorico:.3e} C/kg (Error: {abs(e_m_calc - e_m_teorico)/e_m_teorico*100:.2f}%)')

plt.figure(figsize=(7, 4))
plt.scatter(x_data, U_exp, color='red', s=60, label='Datos Experimentales')
plt.plot(x_data, modelo_lineal(x_data, m_fit), 'b-', lw=2, label=f'Ajuste lineal ($e/m = {e_m_calc:.2e}$ C/kg)')
plt.title('Linealización TP 3: Tensión $U$ vs $(i \cdot r)^2$')
plt.xlabel('$(i \cdot r)^2\ [\mathrm{A^2\,m^2}]$')
plt.ylabel('Tensión de Aceleración $U\ [\mathrm{V}]$')
plt.grid(True, linestyle='--')
plt.legend()
plt.show()

## Práctica 3: Experimento de Franck-Hertz y Picos de Resonancia a $4.9\text{ V}$ (TP 5)
Detección de los picos de corriente anódica debidos a las colisiones inelásticas con átomos de mercurio.

In [ ]:
from scipy.signal import find_peaks

V1 = np.linspace(0, 30, 1000)
# Simulación de señal con picos periódicos cada ~4.9 V y fondo creciente
periodo = 4.9
fondo = 0.05 * (V1**1.5)
modulacion = 2.0 * np.sin(2 * np.pi * V1 / periodo - np.pi/2)
I_anodo = np.maximum(0, fondo + modulacion * (V1 / 10.0))

picos, _ = find_peaks(I_anodo, distance=120, prominence=0.5)
voltajes_picos = V1[picos]
delta_V = np.diff(voltajes_picos)

print('Tensiones de los picos detectados [V]:', np.round(voltajes_picos, 2))
print('Diferencias entre picos consecutivos [V]:', np.round(delta_V, 2))
print(f'Potencial de excitación medio: {np.mean(delta_V):.2f} V (Teórico: 4.90 V)')

plt.figure(figsize=(8, 4.5))
plt.plot(V1, I_anodo, 'b-', lw=2, label='Corriente Anódica $I_A$')
plt.plot(voltajes_picos, I_anodo[picos], 'ro', markersize=8, label='Picos de Excitación')
for vp in voltajes_picos:
    plt.axvline(vp, color='gray', linestyle=':', alpha=0.6)
plt.title('Curva Característica del Experimento de Franck-Hertz (TP 5)')
plt.xlabel('Tensión Aceleradora $V_1$ [V]')
plt.ylabel('Corriente Anódica $I_A$ [u.a.]')
plt.grid(True, linestyle='--')
plt.legend()
plt.show()